In [1]:
import random, uuid

customers_int = 2
ts_step_int = 1

class OrderGenerator:
    def __init__(self):
        self.customer_id_int = 0
        #
        self.ts_int = 0
        self.ts_step_int = ts_step_int

    def generate(self):
        order_id_str = str(uuid.uuid4())
        message_dict = {
            "key": order_id_str,
            "value": {"order_id": order_id_str,
                      "customer_id": random.randint(0, customers_int - 1),
                      "price": random.randint(1, 10000) / 100,
                      "ts": self.ts_int},
        }
        #
        self.ts_int += self.ts_step_int
        #
        return message_dict


In [14]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

order_source_str = "orders"
sink_str = "orders_windowed"
#
tumbling_int = ts_step_int * 100
#
order_tn = (
    Tn.source(order_source_str)
    .expire(lambda x: x["value"]["ts"],
            lambda x: (x["value"]["ts"] // tumbling_int) *  tumbling_int + tumbling_int * 2)
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .distinct()
)
#
group_by_tn = (
    order_tn
    .group_by_agg(
        by_function=lambda x: (
            (x["ts"] // tumbling_int) * tumbling_int + tumbling_int,
            x["customer_id"]
        ),
        select_function=lambda x: x,
        projection_function=lambda by, agg: {
            "window_end": by[0],
            "customer_id": by[1],
            "orders": agg["orders"],
            "total_price": agg["total_price"],
            "last_ts": agg["last_ts"]
        },
        agg_function=lambda agg, x, w: {
            "orders": agg["orders"] + w,
            "total_price": agg["total_price"] + x["price"] * w,
            "last_ts": max(agg["last_ts"], x["ts"]) if w > 0 else agg["last_ts"]
        },
        agg_initial_any={
            "orders": 0,
            "total_price": 0,
            "last_ts": 0
        }
    )
)
#
expel_tn = (
    group_by_tn
    .join(
        order_tn
        .map(lambda x: x["ts"])
        .max(lambda x: x,
             lambda x: {"window_end": (x // tumbling_int) * tumbling_int + tumbling_int}),
        lambda l, r: r["window_end"] > l["window_end"],
        lambda l, _: l
    )
)._peek("expel")
#
built_tn = Tn.build(expel_tn.sink(sink_str))



In [15]:
import cloudpickle

built_tn.reset()
order_generator = OrderGenerator()

for i in range(100):
    print(f"Step {i + 1}")
    order_message_dict_list = [order_generator.generate() for _ in range(100)]
    built_tn.push(order_source_str, order_message_dict_list)
    built_tn.latest()
    print(len(cloudpickle.dumps(built_tn)) / 1024)


Step 1
94.1064453125
Step 2
expel: ({'window_end': 100, 'customer_id': 1, 'orders': 52, 'total_price': 2608.9200000000005, 'last_ts': 99}, 1)
expel: ({'window_end': 100, 'customer_id': 0, 'orders': 48, 'total_price': 2389.8300000000004, 'last_ts': 96}, 1)
160.5537109375
Step 3
expel: ({'window_end': 200, 'customer_id': 0, 'orders': 43, 'total_price': 2259.4199999999996, 'last_ts': 195}, 1)
expel: ({'window_end': 200, 'customer_id': 1, 'orders': 57, 'total_price': 2812.8599999999997, 'last_ts': 199}, 1)
expel: ({'window_end': 100, 'customer_id': 1, 'orders': 52, 'total_price': 2608.9200000000005, 'last_ts': 99}, -1)
expel: ({'window_end': 100, 'customer_id': 0, 'orders': 48, 'total_price': 2389.8300000000004, 'last_ts': 96}, -1)
283.146484375
Step 4
expel: ({'window_end': 300, 'customer_id': 0, 'orders': 49, 'total_price': 2704.82, 'last_ts': 298}, 1)
expel: ({'window_end': 300, 'customer_id': 1, 'orders': 51, 'total_price': 2580.5400000000004, 'last_ts': 299}, 1)
expel: ({'window_end':